## 

# Limpieza de datos

## Introduccion

Antes de realizar cualquier análisis estadístico o generar visualizaciones, es indispensable asegurar que el conjunto de datos se encuentre en condiciones óptimas para su procesamiento. Los registros de defunciones publicados por el INEGI, aunque provienen de una fuente oficial y estructurada, pueden contener elementos que dificultan el análisis directo, como valores faltantes, inconsistencias en los nombres de las entidades federativas, variaciones en los formatos de fecha o códigos, y registros que no corresponden a la causa de interés.

La etapa de limpieza de los datos tiene como propósito transformar el dataset original en una versión depurada, coherente y lista para el análisis. Este proceso permite garantizar la calidad de la información, reducir errores en la interpretación y asegurar que los resultados obtenidos reflejen de manera fiel el comportamiento real de las defunciones por diabetes mellitus tipo 2 en México.

Durante esta fase se aplican procedimientos como la detección y manejo de valores faltantes, la estandarización de categorías, la verificación de la codificación CIE-10 y el filtrado de los registros relevantes. La limpieza de datos constituye un paso fundamental para construir un análisis exploratorio confiable y reproducible, y sienta las bases para obtener conclusiones sólidas en las etapas posteriores del proyecto.

In [18]:
import pandas as pd
import sqlite3
import os

df = pd.read_csv("../data/raw/Defunciones.csv", encoding="latin-1")
df_edad = pd.read_csv("../data/raw/edad.csv", encoding="latin-1")

# renombrar columnas para que coincidan con la base de datos
edad_columns = {
    "CVE": "edad",
    "DESCRIP": "rango_edad"
}
df_edad.rename(columns=edad_columns, inplace=True)

# Filtrar solo diabetes (CIE-10: E10–E14)
codigos_diabetes = df["causa_def"].str.match(r"^E1[0-4]", na=False)
df_diabetes = df[codigos_diabetes].copy()

# Limpiar edades no válidas (INEGI usa 4998 para "no especificado")
df_diabetes = df_diabetes[df_diabetes["edad"] < 4998]

# Mapear claves de entidad a nombres legibles
entidades = {
    "1": "Aguascalientes", 
    "2": "Baja California", 
    "3": "Baja California Sur", 
    "4": "Campeche", 
    "5": "Coahuila de Zaragoza", 
    "6": "Colima", 
    "7": "Chiapas", 
    "8": "Chihuahua", 
    "9": "Ciudad de México",
    "10": "Durango", 
    "11": "Guanajuato", 
    "12": "Guerrero", 
    "13": "Hidalgo", 
    "14": "Jalisco",
    "15": "Estado de México", 
    "16": "Michoacán de Ocampo", 
    "17": "Morelos", 
    "18": "Nayarit", 
    "19": "Nuevo León", 
    "20": "Oaxaca", 
    "21": "Puebla", 
    "22": "Queretaro", 
    "23": "Quintana Roo", 
    "24": "San Luis Potosí", 
    "25": "Sinaloa", 
    "26": "Sonora",
    "27": "Tabasco",
    "28": "Tamaulipas",
    "29": "Tlaxcala",
    "30": "Veracruz de Ignacio de la Llave",
    "31": "Yucatán",
    "32": "Zacatecas",
    "33": "Extranjero"
}



# Mapear las claves de entidad a nombres legibles
df_entidades = pd.DataFrame.from_dict(entidades, orient='index').reset_index()
df_entidades.columns = ["ent_resid", "entidad_nombre"]


# Unir con los nombres de las entidades
df_entidades["ent_resid"] = df_entidades["ent_resid"].astype(int)
df_diabetes = df_diabetes.merge(df_entidades, on="ent_resid", how="left")

#unir con la tabla de edades para obtener rangos de edad
df_diabetes = df_diabetes.merge(df_edad, on="edad", how="left")

#Columnas necesarias para la base de datos
df_diabetes = df_diabetes[["anio_ocur","mes_ocurr", "entidad_nombre", "edad", "sexo", "causa_def","rango_edad"]]  

# limitar a los años de interés (2000-2024)
df_diabetes = df_diabetes[(df_diabetes["anio_ocur"] >= 2000) & (df_diabetes["anio_ocur"] <= 2024)]

# Guardar limpio
df_diabetes.to_csv("../data/processed/defunciones_diabetes.csv", index=False, encoding="utf-8")

C:\Users\mario\AppData\Local\Temp\ipykernel_31968\1247346022.py:5: DtypeWarning: Columns (68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/Defunciones.csv", encoding="latin-1")


## Conclusión de la limpieza del dataset y justificación de la selección de columnas

Tras realizar el proceso de limpieza del dataset original de defunciones, se obtuvo una versión depurada y enfocada en las variables necesarias para el análisis exploratorio de muertes por diabetes mellitus tipo 2. El dataset inicial contenía 819,672 registros y 74 columnas, muchas de ellas relacionadas con niveles territoriales específicos, indicadores administrativos y variables complementarias que no aportaban valor directo al objetivo del proyecto.

Después de aplicar filtros, eliminar redundancias y corregir inconsistencias, el dataset final quedó conformado por 112,561 registros y 7 columnas, lo que representa una reducción significativa que mejora la eficiencia del análisis sin perder información relevante.

## Justificación de las columnas seleccionadas

Las columnas elegidas responden directamente a las necesidades analíticas del proyecto:

+ anio_ocur y mes_ocurr

Permiten estudiar la evolución temporal de las defunciones, identificar tendencias y comparar periodos específicos.
Se filtro el periodio del 2000 - 2024 para un mejor analisis

+ entidad_nombre 

Es esencial para el análisis territorial. La mortalidad por diabetes tipo 2 varía entre estados, por lo que esta columna permite construir comparaciones geográficas y visualizar diferencias regionales.

+ edad

La diabetes tipo 2 está fuertemente asociada con la edad. Esta variable permite analizar qué grupos poblacionales presentan mayor mortalidad y detectar patrones epidemiológicos.

+ Sexo

Permite identificar diferencias entre hombres y mujeres, lo cual es relevante en estudios de salud pública.

+ causa_def  

Es la columna clave para filtrar los casos correspondientes al código CIE‑10 E11, asegurando que el análisis se enfoque exclusivamente en diabetes mellitus tipo 2.

+ rango_edad  

Complementa la variable de edad, permitiendo agrupar los registros en categorías útiles para visualizaciones y análisis descriptivos.

Estas columnas representan el conjunto mínimo necesario para responder las preguntas del proyecto sin introducir ruido analítico.

## Justificación de las columnas añadidas (edades y estados)

Durante la limpieza se añadieron columnas derivadas para mejorar la claridad y utilidad del dataset:

+️ Columnas derivadas de edad

Se generaron variables adicionales para transformar los valores de edad codificados (ej. 4055, 4068) en edades reales y rangos estandarizados. Esto fue necesario porque:

    - Los valores originales representan códigos administrativos, no edades reales.

    - La interpretación epidemiológica requiere edades numéricas claras.

    - Facilita la creación de grupos de edad homogéneos para análisis comparativos.

    - Estas columnas permiten construir visualizaciones más precisas y análisis demográficos confiables.

+️ Columnas derivadas de estados

    - Se añadieron columnas para estandarizar nombres de entidades y corregir inconsistencias. Esto fue necesario porque:

    - Algunos registros contenían variaciones o errores de codificación.

    - La estandarización facilita el análisis territorial y la creación de mapas.

    - Permite agrupar correctamente los registros por entidad federativa.

Estas mejoras garantizan que los análisis por estado sean precisos y reproducibles.

## Conclusión general

El proceso de limpieza permitió transformar un dataset complejo y extenso en una versión clara, manejable y directamente alineada con los objetivos del proyecto. La selección de columnas se justificó en función de su relevancia analítica, mientras que las columnas añadidas fortalecen la calidad del análisis temporal, territorial y demográfico.

El dataset final está listo para:

+ realizar análisis exploratorios,

+ generar visualizaciones significativas,

+ identificar patrones de mortalidad por diabetes tipo 2,

+ construir conclusiones sólidas para el informe.
